In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import IntSlider, HBox, VBox, Output, HTML
from IPython.display import display, clear_output


# ============================================================
# 1. Test Signal (fs = 300 Hz)
# ============================================================

fs = 300  
duration = 1.0
t = np.arange(int(fs * duration)) / fs

x = (
    0.8 * np.sin(2 * np.pi * 50 * t)
    + 0.5 * np.sin(2 * np.pi * 120 * t)
)


# ============================================================
# 2. Parameters
# ============================================================

L0 = 128
R0 = 64
N0 = 128


# ============================================================
# 3. Report & Sliders UI Setup (English with normalization note & red bold conditions)
# ============================================================

report_html = HTML("""
<div style="border: 1px solid #ccc; padding: 10px; border-radius: 5px; background-color: #f9f9f9; font-family: sans-serif; font-size: 13px; line-height: 1.4; margin-bottom: 10px;">
    <b>Overlap-Add (OLA) Analysis & Reconstruction Guide:</b>
    <ul style="margin: 5px 0 0 20px; padding: 0;">
        <li><b>L (Window Length):</b> The length of the window function in samples, determining time resolution.</li>
        <li><b>R (Hop Size):</b> The hop size or step between consecutive frames. It defines the overlap percentage: <span style="color: red; font-weight: bold;">Overlap = 100 × (1 - R/L)%</span>. Note that when <span style="color: red; font-weight: bold;">R = L</span>, the overlap is <span style="color: red; font-weight: bold;">zero (0%)</span>.</li>
        <li><b>N (FFT Size):</b> The number of FFT points. Must satisfy <span style="color: red; font-weight: bold;">N >= L</span> (zero-padding is applied if <span style="color: red; font-weight: bold;">N > L</span>).</li>
        <li><b>Condition for Perfect Reconstruction:</b> To achieve flawless signal reconstruction without amplitude modulation or aliasing, the sum of the overlapping windows must be a constant value across time. For standard windows, this constant sum condition is typically met when <span style="color: red; font-weight: bold;">R <= L/2</span> (e.g., <span style="color: red; font-weight: bold;">50% overlap or more</span>).</li>
        <li><b>Signal Normalization:</b> To eliminate amplitude scaling fluctuations caused by overlapping windows in practice, we perform normalization by dividing the raw OLA reconstruction by the accumulated window sum (<span style="color: red; font-weight: bold;">window_sum</span>). This effectively removes window distortion and restores the original signal's true amplitude.</li>
    </ul>
</div>
""")

L_slider = IntSlider(
    value=L0,
    min=32,
    max=256,
    step=16,
    description='L:',
    continuous_update=True,
    style={'description_width': 'initial'},
    layout={'width': '350px'}
)

R_slider = IntSlider(
    value=R0,
    min=8,
    max=256,
    step=8,
    description='R:',
    continuous_update=True,
    style={'description_width': 'initial'},
    layout={'width': '350px'}
)

N_slider = IntSlider(
    value=N0,
    min=32,
    max=512,
    step=16,
    description='N:',
    continuous_update=True,
    style={'description_width': 'initial'},
    layout={'width': '350px'}
)


# ============================================================
# 4. Output
# ============================================================

plot_output = Output()


# ============================================================
# 5. Overlap-Add Function (με ενσωματωμένο το γράφημα επιμέρους παραθύρων)
# ============================================================

def update(L, R, N):

    with plot_output:
        clear_output(wait=True)

        if R > L:
            print('Choose R <= L.')
            return

        if N < L:
            print('Choose N >= L.')
            return

        window = signal.windows.hamming(L, sym=False)

        starts = np.arange(0, len(x) - L + 1, R)

        reconstructed = np.zeros(len(x))
        window_sum = np.zeros(len(x))

        for start in starts:

            frame = x[start:start + L] * window

            spectrum = np.fft.fft(frame, n=N)

            frame_rec = np.fft.ifft(spectrum).real[:L]

            reconstructed[start:start + L] += frame_rec
            window_sum[start:start + L] += window

        normalized = np.divide(
            reconstructed,
            window_sum,
            out=np.zeros_like(reconstructed),
            where=window_sum > 1e-12
        )

        overlap = 100 * (1 - R / L)

        # Δημιουργία 4 υπογραφήμων (ax1: επιμέρους παράθυρα, ax2: άθροισμα, ax3: raw ola, ax4: normalized)
        fig, (ax1, ax2, ax3, ax4) = plt.subplots(
            4, 1,
            figsize=(17.25, 11),
            gridspec_kw={'height_ratios': [2, 2, 2, 2]}
        )

        # ----------------------------------------------------
        # 1. Individual overlapping windows (από το 1ο notebook)
        # ----------------------------------------------------
        n_indices = np.arange(L)
        for start in starts:
            ax1.plot(start + n_indices, window, linewidth=1.5)
            ax1.fill_between(start + n_indices, 0, window, alpha=0.1)

        ax1.set_xlim(-10, len(x) + 10)
        ax1.set_ylim(0, 1.1)
        ax1.set_title(
            f'Individual Hamming Windows in Time    '
            f'(L = {L}, R = {R}, Overlap = {overlap:.1f}%)',
            fontsize=11
        )
        ax1.set_ylabel('Amplitude')
        ax1.grid(True, alpha=0.3)

        # ----------------------------------------------------
        # 2. Window sum
        # ----------------------------------------------------

        ax2.plot(window_sum, color='black', label='Window sum')

        ax2.set_title(
            'Sum of Overlapping Hamming Windows (window_sum)',
            fontsize=11
        )

        ax2.set_ylabel(r'$\bar{w}[n]$')
        ax2.grid(True, alpha=0.3)
        ax2.legend(loc='center left', bbox_to_anchor=(1, 0.5))

        # ----------------------------------------------------
        # 3. Raw overlap-add
        # ----------------------------------------------------

        ax3.plot(x, color='black', alpha=0.5, label='Original signal')
        ax3.plot(
            reconstructed,
            color='red',
            alpha=0.8,
            label='Raw overlap-add'
        )

        ax3.set_title(
            'Overlap-Add Reconstruction Before Normalization',
            fontsize=11
        )

        ax3.set_ylabel('Amplitude')
        ax3.grid(True, alpha=0.3)
        ax3.legend(loc='center left', bbox_to_anchor=(1, 0.5))

        # ----------------------------------------------------
        # 4. Normalized reconstruction
        # ----------------------------------------------------

        ax4.plot(x, color='black', alpha=0.5, label='Original signal')
        ax4.plot(
            normalized,
            color='red',
            alpha=0.8,
            label='Reconstructed signal'
        )

        ax4.set_title(
            'Overlap-Add Reconstruction After Normalization',
            fontsize=11
        )

        ax4.set_xlabel('Sample')
        ax4.set_ylabel('Amplitude')
        ax4.grid(True, alpha=0.3)
        ax4.legend(loc='center left', bbox_to_anchor=(1, 0.5))

        plt.tight_layout()
        display(fig)
        plt.close(fig)


# ============================================================
# 6. Dynamic Constraints & Callbacks
# ============================================================

def on_L_change(change):
    if change['name'] == 'value':
        new_L = change['new']
        
        # R constraint: R <= L
        if R_slider.value > new_L:
            R_slider.value = new_L
        R_slider.max = new_L
        
        # N constraint: N >= L
        if N_slider.value < new_L:
            N_slider.value = new_L
        N_slider.min = new_L
        
        update(L_slider.value, R_slider.value, N_slider.value)

def on_R_change(change):
    if change['name'] == 'value':
        if R_slider.value > L_slider.value:
            R_slider.value = L_slider.value
        else:
            update(L_slider.value, R_slider.value, N_slider.value)

def on_N_change(change):
    if change['name'] == 'value':
        if N_slider.value < L_slider.value:
            N_slider.value = L_slider.value
        else:
            update(L_slider.value, R_slider.value, N_slider.value)

L_slider.observe(on_L_change, names='value')
R_slider.observe(on_R_change, names='value')
N_slider.observe(on_N_change, names='value')


# ============================================================
# 7. Initial Display
# ============================================================

update(L0, R0, N0)

display(
    VBox([
        report_html,
        HBox([L_slider, R_slider, N_slider]),
        plot_output
    ])
)